In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("init_load_flag", "")
init_load_flag = int(dbutils.widgets.get("init_load_flag"))

In [0]:
df = spark.read.table("databricks_cata.silver.products_silver")

In [0]:
df = df.drop("discounted_price")

In [0]:
df.limit(5).display()

In [0]:
df = df.withColumn("row_hash", md5(concat_ws("||", col("product_name"), col("brand"), col("category"), col("price"))))

In [0]:
df.limit(5).display()

In [0]:
if init_load_flag == 0:
    df_old = spark.sql('''select DimProductKey, product_id, row_hash, create_date, update_date
                       from databricks_cata.gold.DimProductsscd2
                       where is_current = 1''')

else:
    df_old = spark.sql('''select 0 DimProductKey, 0 product_id,0 row_hash, 0 create_date, 0 update_date
                       from databricks_cata.silver.products_silver
                       where 1=2''')

In [0]:
df_old.display()

In [0]:
df_old = df_old\
    .withColumnRenamed("DimProductKey", "old_DimProductKey")\
    .withColumnRenamed("product_id", "old_product_id")\
    .withColumnRenamed("row_hash", "old_row_hash")\
    .withColumnRenamed("create_date", "old_create_date")\
    .withColumnRenamed("update_date", "old_update_date")

In [0]:
df_join = df.join(df_old, df.product_id == df_old.old_product_id,"left")

In [0]:
df_join.limit(5).display()

In [0]:
df_new = df_join.filter(df_join.old_DimProductKey.isNull())
df_new.display()

In [0]:
df_changed = df_join.filter((df_join.old_DimProductKey.isNotNull()) & (df_join.row_hash != df_join.old_row_hash))
df_changed.display()

In [0]:
df_unchanged = df_join.filter(
    df_join.old_DimProductKey.isNotNull() & (df_join.row_hash == df_join.old_row_hash)
)
print("unchanged count (no action needed):", df_unchanged.count())

In [0]:
df_expire = df_changed.select(
    col("old_DimProductKey").alias("DimProductKey")
).withColumn("merge_key", col("DimProductKey").cast("string"))

In [0]:
df_changed_new_version = df_changed.select(
    df_changed.product_id,
    df_changed.product_name,
    df_changed.category,
    df_changed.brand,
    df_changed.price,
    df_changed.row_hash   
)

df_changed_new_version = df_changed_new_version\
    .withColumn("effective_start_date", current_timestamp())\
    .withColumn("effective_end_date", lit(None).cast("timestamp"))\
    .withColumn("is_current", lit(1))\
    .withColumn("create_date", current_timestamp())\
    .withColumn("update_date", current_timestamp())

In [0]:
df_changed_new_version.display()

In [0]:
df_new = df_new.select(
    df_new.product_id,
    df_new.product_name,
    df_new.category,
    df_new.brand,
    df_new.price,
    df_new.row_hash   
)

df_new = df_new\
    .withColumn("effective_start_date", current_timestamp())\
    .withColumn("effective_end_date", lit(None).cast("timestamp"))\
    .withColumn("is_current", lit(1))\
    .withColumn("create_date", current_timestamp())\
    .withColumn("update_date", current_timestamp())
df_new.display()


In [0]:
df_inserts = df_new.unionByName(df_changed_new_version)
df_inserts.limit(5).display()

In [0]:
# df_inserts = df_inserts.withColumn("DimProductKey", monotonically_increasing_id() + lit(1))

w = Window.partitionBy(lit(1)).orderBy("product_id")
df_inserts = df_inserts.withColumn("DimProductKey", row_number().over(w))

In [0]:
if init_load_flag == 1:
    max_surrogate_key = 0
else:
    df_maxsur = spark.sql("select max(DimProductKey) as max_surrogate_key from databricks_cata.gold.DimProductsscd2")
    max_surrogate_key = df_maxsur.collect()[0]['max_surrogate_key']
    if max_surrogate_key is None:
        max_surrogate_key = 0

In [0]:
df_inserts = df_inserts.withColumn("DimProductKey", lit(max_surrogate_key) + col("DimProductKey"))
df_inserts = df_inserts.withColumn("merge_key", lit(None).cast("string"))
df_inserts.limit(5).display()

In [0]:
expire_cols = [c for c in df_inserts.columns if c not in ("DimProductKey", "merge_key")]
df_expire_padded = df_expire
for c, dtype in df_inserts.select(*expire_cols).dtypes:
    df_expire_padded = df_expire_padded.withColumn(c, lit(None).cast(dtype))

df_merge_source = df_inserts.select("DimProductKey", "merge_key", *expire_cols)\
    .unionByName(df_expire_padded.select("DimProductKey", "merge_key", *expire_cols))

df_merge_source.display()

In [0]:
if spark.catalog.tableExists("databricks_cata.gold.DimProductsscd2"):
    dlt_obj = DeltaTable.forPath(spark, "abfss://gold@customersprojectete.dfs.core.windows.net/DimProductsscd2")

    dlt_obj.alias("tgt").merge(
        df_merge_source.alias("src"),
        "tgt.DimProductKey = src.merge_key"
    ).whenMatchedUpdate(set={
        "is_current": lit(0),
        "effective_end_date": current_timestamp(),
        "update_date": current_timestamp()
    }).whenNotMatchedInsertAll().execute()

else:
    df_inserts.drop("merge_key").write.mode("overwrite")\
        .format("delta")\
        .option("path", "abfss://gold@customersprojectete.dfs.core.windows.net/DimProductsscd2")\
        .saveAsTable("databricks_cata.gold.DimProductsscd2")

In [0]:
%sql
select * from databricks_cata.gold.DimProductsscd2 order by product_id, effective_start_date


In [0]:
%sql
INSERT INTO databricks_cata.silver.products_silver
VALUES ('P9999', 'Air Max 90', 'Footwear', 'Nike', 2999.99)

In [0]:
%sql
UPDATE databricks_cata.silver.products_silver
SET category = 'Premium Footwear'
WHERE product_id = 'P0001'

In [0]:
%sql
-- ALTER TABLE databricks_cata.silver.products_silver SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');

In [0]:
%sql
-- ALTER TABLE databricks_cata.silver.products_silver DROP COLUMN discounted_price
;

In [0]:
%sql
-- select * from databricks_cata.silver.products_silver where 1=5

In [0]:
# %sql
# INSERT INTO databricks_cata.silver.products_silver
# VALUES (9999, 'Test Product', 'Electronics', 'TestBrand', 499.99)

In [0]:
# print(df_merge_source.columns)

In [0]:
# spark.sql("DROP TABLE IF EXISTS databricks_cata.gold.DimProductsscd2")

# dbutils.fs.rm("abfss://gold@customersprojectete.dfs.core.windows.net/DimProductsscd2", recurse=True)

In [0]:
# %sql
# UPDATE databricks_cata.gold.DimProductsscd2 
# SET category = "original_category_value"
# WHERE product_id = "P0001"

In [0]:
# %sql
# UPDATE databricks_cata.silver.products_silver
# SET category = "gurling"
# WHERE product_id = "P0001"